# OrthoGenesisAI — Cloud Training (Colab Pro)

Trains `XRayTo3DNet` on the CTPelvic1K dataset using a Colab GPU.

**Before running:** Runtime → Change runtime type → Hardware: **GPU** → Pick **A100** if available, else L4, else T4.

Make sure your Google Drive has `MyDrive/OrthoGenesisAI/` with the `backend/` folder uploaded into it.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/OrthoGenesisAI'
assert os.path.isdir(PROJECT_DIR), f'Missing: {PROJECT_DIR}'
print('OK. Contents:', os.listdir(PROJECT_DIR))

## 3. Install dependencies

In [ ]:
!pip install -q nibabel scikit-image trimesh

## 4. Copy code to local disk (faster I/O than Drive); link the dataset

In [ ]:
import shutil, os

LOCAL_ROOT = '/content/orthogenesis'
if os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
os.makedirs(LOCAL_ROOT)

shutil.copytree(f'{PROJECT_DIR}/backend/app', f'{LOCAL_ROOT}/app')

os.makedirs(f'{LOCAL_ROOT}/data', exist_ok=True)
os.symlink(f'{PROJECT_DIR}/backend/data/ct_processed', f'{LOCAL_ROOT}/data/ct_processed')

CKPT_DIR = f'{PROJECT_DIR}/checkpoints_colab'
os.makedirs(CKPT_DIR, exist_ok=True)
os.symlink(CKPT_DIR, f'{LOCAL_ROOT}/data/checkpoints')
print('Setup complete.')

## 5. Patch `train.py` to use CUDA (it's hardcoded to CPU)

In [ ]:
train_py = f'{LOCAL_ROOT}/app/reconstruction/train.py'
src = open(train_py).read()
src = src.replace('device = torch.device("cpu")',
                  'device = torch.device("cuda" if torch.cuda.is_available() else "cpu")')
open(train_py, 'w').write(src)
print('Patched.')

## 6. Train

Hyperparameters scaled up vs CPU defaults:
- `feat-dim 128` (was 64) → 4× model capacity
- `image-res 224`, `volume-res 96`, `n-points 16384`
- `batch-size 8`, `epochs 100`

Expected checkpoint size: 80–150 MB. Total training time: ~6–12 hrs on A100, ~24 hrs on T4.

In [ ]:
%cd /content/orthogenesis
!python -m app.reconstruction.train \
    --nifti-dir data/ct_processed \
    --epochs 100 \
    --batch-size 8 \
    --feat-dim 128 \
    --volume-res 96 \
    --image-res 224 \
    --n-points 16384 \
    --lr 1e-3 \
    --workers 4

## 7. Verify checkpoint

In [ ]:
import os, torch
ckpt = f'{PROJECT_DIR}/checkpoints_colab/best.pt'
print(f'Size: {os.path.getsize(ckpt)/1e6:.1f} MB')
state = torch.load(ckpt, map_location='cpu', weights_only=False)
print(f'Epoch: {state["epoch"]}  val_loss: {state["val_loss"]:.4f}')
print(f'Config: {state["model_config"]}')

## 8. Download checkpoint to local Mac

In [ ]:
from google.colab import files
files.download(f'{PROJECT_DIR}/checkpoints_colab/best.pt')